# Sherlock Holmes: a language model from scratch

Last time: Word2Vec learned word representations. Today: learn
$P(x_{t+1}\mid x_{\leq t})$ and generate text, one character at a time.

## Setup

In [ ]:
import math
import random
import re
import time
import urllib.request
import base64
import zlib
import unicodedata
import textwrap

import torch
from torch import nn
from torch.nn import functional as F

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
BATCH_SIZE = 32
SEQ_LEN = 64
HIDDEN_SIZE = 192
LEARNING_RATE = 0.003
TRAIN_STEPS = 4000
LOG_EVERY = 500
EVAL_BATCHES = 8
print("PyTorch:", torch.__version__, "| device:", DEVICE)


In [ ]:
def download_book(timeout=20):
    headers = {"User-Agent": "Mozilla/5.0 (educational SGNS notebook)"}
    try:
        request = urllib.request.Request(
            'https://www.gutenberg.org/cache/epub/1661/pg1661.txt', headers=headers
        )
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw = response.read().decode("utf-8", errors="replace")
        if len(raw) < 100_000 or "Sherlock Holmes" not in raw:
            raise ValueError("The response is not the expected book.")
        print(f"Downloaded {len(raw):,} characters from Project Gutenberg.")
        return raw, "Project Gutenberg: The Adventures of Sherlock Holmes"
    except Exception as exc:
        print(f"Download unavailable ({type(exc).__name__})")


def strip_gutenberg_boilerplate(text):
    start_match = re.search(r"\*\*\* START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.I)
    end_match = re.search(r"\*\*\* END OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.I)
    start = start_match.end() if start_match else 0
    end = end_match.start() if end_match else len(text)
    return text[start:end]


def clean_text(text):
    text = text.translate(str.maketrans({"“": '"', "”": '"', "‘": "'", "’": "'",
                                       "—": "-", "–": "-", "…": "..."}))
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", text).strip()


raw_text, corpus_source = download_book()
text = clean_text(strip_gutenberg_boilerplate(raw_text))
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
VOCAB_SIZE = len(chars)

def encode(s):
    return torch.tensor([stoi[ch] for ch in s], dtype=torch.long, device=DEVICE)

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

data = encode(text)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
assert len(val_data) > SEQ_LEN
print(corpus_source)
print(f"Characters: {len(data):,} | vocabulary: {VOCAB_SIZE}")
print(f"Train: {len(train_data):,} | validation: {len(val_data):,}")
print("Vocabulary:", repr("".join(chars)))
print(text[1000:1300])


## Dataset
Each target is shifted by one character.

In [ ]:
def get_batch(source, batch_size=BATCH_SIZE, seq_len=SEQ_LEN, generator=None):
    starts = torch.randint(len(source) - seq_len, (batch_size,), generator=generator)
    positions = starts[:, None] + torch.arange(seq_len + 1)[None, :]
    windows = source[positions]                 # (B, S+1)
    return windows[:, :-1], windows[:, 1:]     # both (B, S), integer IDs

train_rng = torch.Generator().manual_seed(SEED)
x_batch, y_batch = get_batch(train_data, generator=train_rng)
print("Input IDs:", tuple(x_batch.shape), "| targets:", tuple(y_batch.shape))
print("One-hot input:", tuple(F.one_hot(x_batch, VOCAB_SIZE).shape))
print("input: ", decode(x_batch[0]))
print("target:", decode(y_batch[0]))
assert torch.equal(x_batch[:, 1:], y_batch[:, :-1])


## TASK I. the vanilla RNN
$$h_t=\tanh(x_tW_{xh}+h_{t-1}W_{hh}+b_h),\qquad z_t=h_tW_{hy}+b_y.$$
Here $x_t$ is one-hot; $z_t$ contains **logits**, not probabilities.

Implement parameters, zero initial state, one step, and the sequence loop.

| Tensor | Shape |
|---|---|
| One-hot input $x_t$ | `(B, V)` |
| Hidden state $h_t$ | `(B, H)` |
| $W_{xh}, W_{hh}, W_{hy}$ | `(V, H)`, `(H, H)`, `(H, V)` |
| All logits | `(B, S, V)` |

**Hints:** use `nn.Parameter`, `@`, `torch.tanh`, and `torch.stack(..., dim=1)`.
Reuse the **same** matrices at every time step. Return both logits and the final hidden state.


In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        #
        # ### Your code here ###
        #

        # Model

        # Init

    def step(self, x_t, h):

        #
        # ### Your code here ###
        #
        
        return logits, h

    def forward(self, ids, h=None):
        # ids: (B, S), integer character IDs

        #
        # ### Your code here ###
        #

        if h is None:
            # Init h   


        # Return LOGITS

In [ ]:
# Check both shapes and hidden-state continuity.
model = VanillaRNN(VOCAB_SIZE, HIDDEN_SIZE).to(DEVICE)
with torch.no_grad():
    logits, h = model(x_batch)
    left, h_left = model(x_batch[:, :20])
    right, _ = model(x_batch[:, 20:], h_left)
assert logits.shape == (BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)
assert h.shape == (BATCH_SIZE, HIDDEN_SIZE)
torch.testing.assert_close(logits, torch.cat([left, right], dim=1))
print("Logits:", tuple(logits.shape), "| hidden:", tuple(h.shape))
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))


## TASK II. loss and perplexity
$$L=-\frac{1}{T}\sum_t\log P(x_{t+1}\mid x_{\le t}),\qquad \mathrm{PPL}=e^L.$$
We average over all **batch × sequence** targets. Logs are natural logs.
Perplexity is approximately an **effective branching factor**: a uniform predictor over $V$ characters has PPL $V$.
Lower validation PPL means better prediction on this held-out text, not necessarily better stories.

**Hints:** flatten `(B, S, V)` to `(B*S, V)` and targets to `(B*S,)`.
Use `F.cross_entropy` on raw logits (it already handles log-softmax).
Return a differentiable loss and a detached numeric perplexity.


In [ ]:
def loss_and_perplexity(logits, targets):
    return loss, perplexity

## Training
**BPTT:** autograd propagates gradients backward through the unrolled recurrent computation.
Repeated Jacobian products can shrink or amplify gradients: vanishing/exploding gradients.
Gradient clipping limits explosions; it does not solve vanishing gradients.

Each sampled window starts with zero hidden state. We backpropagate through its 64 steps;
carrying hidden state between unrelated random windows would mix unrelated contexts.
Validation uses fixed sampled windows, at temperature 1, with no gradient updates.
Start training now; read Task 3 while it runs.


In [ ]:
eval_rng = torch.Generator().manual_seed(SEED + 1)
val_batches = [get_batch(val_data, generator=eval_rng) for _ in range(EVAL_BATCHES)]

@torch.no_grad()
def evaluate(model):
    was_training = model.training
    model.eval()
    total_nll, total_chars = 0.0, 0
    for xb, yb in val_batches:
        logits, _ = model(xb)
        batch_loss, _ = loss_and_perplexity(logits, yb)
        total_nll += batch_loss.item() * yb.numel()
        total_chars += yb.numel()
    model.train(was_training)
    mean_loss = total_nll / total_chars
    return mean_loss, math.exp(mean_loss)

torch.manual_seed(SEED)
train_rng.manual_seed(SEED)
model = VanillaRNN(VOCAB_SIZE, HIDDEN_SIZE).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = []
initial_val_loss, initial_val_ppl = evaluate(model)
print(f"Before training: val loss={initial_val_loss:.3f}, val PPL={initial_val_ppl:.2f}")
started = time.perf_counter()
running_loss, running_steps = 0.0, 0
model.train()
for step in range(1, TRAIN_STEPS + 1):
    xb, yb = get_batch(train_data, generator=train_rng)
    optimizer.zero_grad(set_to_none=True)
    logits, _ = model(xb)
    loss, _ = loss_and_perplexity(logits, yb)
    if not torch.isfinite(loss):
        raise RuntimeError("Non-finite loss; try a smaller learning rate.")
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0, error_if_nonfinite=True)
    optimizer.step()
    running_loss += loss.item()
    running_steps += 1
    if step % LOG_EVERY == 0 or step == TRAIN_STEPS:
        train_loss = running_loss / running_steps
        val_loss, val_ppl = evaluate(model)
        elapsed = time.perf_counter() - started
        history.append(dict(step=step, train_loss=train_loss, val_loss=val_loss,
                            val_ppl=val_ppl, seconds=elapsed))
        print(f"{step:4d}/{TRAIN_STEPS} | train loss {train_loss:.3f}, PPL {math.exp(train_loss):.2f}"
              f" | val loss {val_loss:.3f}, PPL {val_ppl:.2f} | {elapsed:.1f}s")
        running_loss, running_steps = 0.0, 0
print(f"Training finished in {(time.perf_counter() - started):.1f} seconds.")


## TASK III. generate autoregressively
1. Feed the **whole prompt once** to build its hidden state.
2. Use its final logits to sample the next character:
   $$p=\operatorname{softmax}(z/T),\qquad x_{\mathrm{new}}\sim p.$$
3. Feed the **sampled character itself** back, keeping the hidden state. Repeat.

**Hints:** use `torch.no_grad()`, `logits[:, -1, :]`, `torch.multinomial`, and `model(next_id, h)`.
The last prompt character is already consumed; do not feed it twice.
Support any nonempty prompt made from vocabulary characters; report unsupported characters clearly.


In [ ]:
@torch.no_grad()
def generate(model, prompt="Sherlock Holmes", max_new_chars=400, temperature=0.8, seed=123):
    model.eval()

    prompt_ids = encode(prompt).unsqueeze(0)     # (1, prompt_length)

    #
    # ### Your code here ###
    #

    # Feed the prompt

    # Generate character (id) one at a time

    return prompt + decode(generated_ids)

### Experiments

In [ ]:
sample = generate(model, "Sherlock Holmes", max_new_chars=600, temperature=0.8)
print(textwrap.fill(sample, width=100))

In [ ]:
for temperature in (0.3, 0.8, 1.3):
    print(f"\nTEMPERATURE = {temperature}")
    print(textwrap.fill(generate(model, "Sherlock Holmes", max_new_chars=350,
                                 temperature=temperature, seed=123), width=100))